# 🫀 퀘스트 46 · Q7-E — **긴RR 앵커 전수 적용(anchor-sweep)**

| | **MedKOS / `notebooks/quest46_q7e_anchor_sweep.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0054`(Q7-D) · `ailab-2026-0053`(Q7-B′) |
| 규약 | **R11-c · R15 · R15-d · R16 · R17 · R19** |
| 학습 | **0회** — `_medref` 를 갈아끼우고 거리 특징만 다시 계산한다 |

## 이 실험이 답해야 하는 것

Q7-D 에서 `#865`(S 57.6%) 하나에 대해 이걸 봤다.

| 기준 | P영역 거리 AUROC |
|---|---|
| 다수결(`_medref`) | **0.2103** ← 뒤집힘 |
| 무감독·큰 군 | 0.0405 ← 더 뒤집힘 |
| 오라클(N군 중앙, 라벨 사용) | 0.9667 |
| **무감독·긴RR 군(앵커)** | **0.9666** |

**한 개체에서** 앵커가 오라클을 따라잡았다. 이제 물어야 할 건 하나다 —
**그 앵커를 전수 72개체에 걸면 정상 개체를 망가뜨리지 않는가.**
반전 개체를 살리려고 나머지 71개를 깎아먹으면 처방이 아니다.

> ⚠️ **비교 대상을 헷갈리면 안 된다.** 여기서 재는 건 **무감독 거리 특징**이지
> Q7-B 의 **학습된 모델**이 아니다. 이 노트북의 매크로를 **0.8842 와 나란히 놓지 않는다**.
> 유효한 비교는 **같은 특징족 안에서** `_medref` 기준 ↔ 앵커 기준, 그것도 **짝지어서**다.

## 사전등록

| 관문 | 내용 | 문턱 |
|---|---|---|
| **E1** | **개선** — 짝지은 (앵커 − 다수결) 매크로 차 | 부트스트랩 CI **하한 > 0** |
| **E2** | **반전 개체 0** — 앵커 기준에서 **유의하게** 뒤집힌 개체 수 (개체별 부트스트랩 SE + 본페로니) | **= 0** |
| **E3** | **정상 개체 비열등** — 유병률 ≤ 0.5 개체에서 (앵커 − 다수결) | CI 하한 **> −0.02** |
| **E4** | **오라클 회복률** — (오라클 − 앵커) 매크로 차 | CI 상한 **< 0.05** |
| **E5** | **RR 위치형 대비 비열등** — (앵커형태 − RR위치형) | CI 하한 **> −0.05** |

**E1·E2·E3 이 처방의 성립 조건**이다. E4 는 상한까지 얼마나 갔는지, E5 는
**"형태로 갈아타면 되는 거냐"** 에 대한 답이다.

> ⚠️ **E1 은 설계상 검정력이 낮다.** SVDB 에서 유병률 > 0.5 인 개체는 `#865` **하나**다
> (Q7-D 【D-B】). 개체를 재표집하는 부트스트랩에서 그 하나가 빠지는 재표집이 흔하므로
> CI 하한이 0 을 넘기 어렵다. **E1 기각/미결을 '앵커가 쓸모없다' 로 읽지 않는다** —
> 그건 반전 개체가 하나뿐이라는 코호트 사실이다. **성립 여부는 E2·E3 이 판정**한다.

### 픽스처가 미리 잡은 것 — **점추정 문턱은 n 을 모른다**

처음엔 E2 를 「방향 AUROC < 0.5 이고 정보량 ≥ 0.60」으로 잡았다. 합성 **null 코호트**
(S 와 N 의 형태를 아예 같게 만든 것)를 돌렸더니, S 가 16비트뿐인 개체가 우연히
**AUROC 0.344**(정보량 0.656)로 찍혀 '반전' 에 잡혔다. **신호가 0인 코호트인데도.**
→ 개체별 부트스트랩 CI 로 바꿨다. **그래도 걸렸다** — CI 상한 0.491. 개체마다 검정을
반복하니 14개면 약 0.35개가 우연히 걸리는 게 정상이다.
→ 최종: **본페로니 보정**(양측 α/n)한 문턱 `AUROC + z·SE < 0.5`. 점추정 반전은 참고로 병기.
**관문을 두 번 고쳐 쓴 건 데이터를 보기 전, 합성 null 위에서다** — 실측 뒤 조정이 아니다.

### 미리 밝혀두는 실패 가능성 — 앵커는 **리듬 사전지식**이다

긴RR 앵커가 쓰는 유일한 가정은 **「이소성 박동은 이르다」**다. 이건 형태 지식이 아니라
**리듬 지식**이다. 그래서 **늦은 이소성**(접합부 이탈 `j`, 심실 이탈 `E`)이 섞인 개체에서는
앵커가 **거꾸로 된 군을 기저로 고른다**. 픽스처가 이 시나리오를 합성해서
**관문이 실제로 기각하는지** 확인한다(무조건 통과기가 아님을 보이는 역방향 검정).

### 채점에서 빠지는 개체는 **이름으로** 남긴다 (R16 · R17)

- 비트 < `MIN_BEATS`, S < `GMIN`, 전부 S → 채점 불가
- k-means 두 군 중 작은 쪽 < `MIN_CLUS`, 또는 두 군의 중앙 RR 동률 → **앵커 퇴화**

조용히 빼면 매크로가 좋아 보인다. 빠진 개체는 전부 사유와 함께 출력한다.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0   = 20260803
GMIN    = 2          # Q7-B 승계 · 여기서 다시 고르지 않는다
FOCUS   = 865        # Q7-D 의 반전 개체 (그림 주석용)
IDX_S   = 1
NB_BOOT = 4000       # 개체를 재표집하는 **레코드 수준** 부트스트랩
NB_REC  = 400        # 비트를 재표집하는 **레코드 내부** 부트스트랩(개체별 CI)
P_SEG   = (0, 85)    # P 파 영역(R 앞 ~236ms @360Hz) — **주지표 축**
QRS_SEG = (85, 130)
FULL_SEG = (0, 300)  # 부지표

# ── 사전등록 상수. **이 아래 어느 셀에서도 다시 고르지 않는다.**
MIN_BEATS = 30       # k-means 2군을 세우려면 최소한
MIN_CLUS  = 5        # 작은 군이 이보다 작으면 앵커 퇴화
INFO_MIN  = 0.60     # (참고 표시용) 점추정 반전을 셀 때의 신호 문턱 — Q7-D 승계
#   ★ 픽스처가 잡은 것: **점추정에 고정 문턱을 걸면 n 이 작은 개체에서 오발한다.**
#     합성 null 코호트(S/N 형태가 아예 같음)에서 S 16비트짜리 개체가 AUROC 0.344 로
#     찍혔다 — 정보량 0.656 ≥ 0.60 을 통과해 '반전' 으로 세어졌다. 신호는 0인데.
#     → **관문 E2 는 개체별 부트스트랩 CI 상한 < 0.5** 로 판정한다(표본수를 안다).
#       INFO_MIN 기반 점추정 반전은 **참고로 병기**만 한다.
NI_NORMAL = 0.02     # E3 비열등 여유
ORACLE_MAX = 0.05    # E4 오라클 격차 상한
NI_RR     = 0.05     # E5 비열등 여유

CONFIG = dict(
    exp="quest46_q7e_anchor_sweep", quest="ailab-2026-0046", step="svdb-anchor-sweep",
    parent_exp=["quest46_q7d_inversion", "ailab-2026-0054"],
    purpose=("Q7-D 가 #865 한 개체에서 찾은 **긴RR 앵커**를 전수 개체에 걸어, "
             "반전을 없애면서 **정상 개체를 망가뜨리지 않는지**를 짝지어 잰다"),
    dataset="SVDB 전수 · Q7-B 예측 캐시(라벨·매핑용) + svdb_data5.npz (학습 0회)",
    gmin=GMIN, min_beats=MIN_BEATS, min_clus=MIN_CLUS, info_min=INFO_MIN,
    predictions={
        "E1": "짝지은 (앵커 − 다수결) 매크로 차의 CI 하한 > 0",
        "E2": "앵커 기준에서 **유의하게** 반전된 개체(개체별 부트스트랩 CI 상한 < 0.5) 수 = 0",
        "E3": f"유병률 ≤ 0.5 개체에서 (앵커 − 다수결) CI 하한 > −{NI_NORMAL}",
        "E4": f"(오라클 − 앵커) 매크로 차 CI 상한 < {ORACLE_MAX}",
        "E5": f"(앵커 형태 − RR 위치형) CI 하한 > −{NI_RR}"},
    caveat=("**이 노트북의 매크로는 무감독 거리 특징의 값**이다 — Q7-B 학습 모델의 "
            "0.8842 와 나란히 놓지 않는다(다른 채점기). 유효한 비교는 같은 특징족 안의 "
            "**짝지은 차**뿐이다. 오라클은 라벨을 쓰므로 성능이 아니라 **상한**이다. "
            "앵커의 가정 「이소성은 이르다」는 **리듬 사전지식**이며 늦은 이소성"
            "(이탈박동)에서는 틀린다 — 픽스처 역방향 검정 참조. "
            "**E1 은 설계상 저검정력**(유병률>0.5 개체가 SVDB 에 1개뿐) — 성립은 E2·E3 이 판정."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7e_anchor_sweep", CONFIG, project=PROJECT)
run.log("설정 ✅ 학습 0회 · 주지표 축 = P영역 거리 · 비교는 **짝지은 차**")

In [ ]:
# CELL 2 — 【G0】 자산 · 매핑 (Q7-D 와 동일 규약 — fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
CNTJ = os.path.join(PROJECT, "data", "svdb_ann_counts.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (CNTJ, "Q7-A 주석 카운트"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
CNT = {int(k): {int(kk): vv for kk, vv in v.items()} for k, v in json.load(open(CNTJ)).items()}
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다 — 연속 번호로 대체하지 않는다")

labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
EXCL17 = sorted(MAP[l] for l in labels if l not in recs)
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels) and not (set(REC.tolist()) - set(recs))

CNT3 = {r: (v.get(0, 0), v.get(1, 0), v.get(2, 0)) for r, v in CNT.items()}
mism = []
for r in np.unique(REC):
    o = tuple(int((Y[REC == r] == k).sum()) for k in range(3)); a = CNT3.get(int(r))
    if not a or any(abs(x - y) > 2 for x, y in zip(o, a)):
        mism.append((int(r), o, a))
run.log("\n" + "=" * 100)
run.log("【G0】 자산 · 매핑")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(np.unique(REC))}개 · Q7-B 가 놓쳤던 {len(EXCL17)}개: {EXCL17}")
run.log(f"  (N,S,V) 재검증 불일치 **{len(mism)}건**")
for r, o, a in mism:
    run.log(f"    #{r}  예측 {o}  vs 주석 {a}   차 {tuple(x-y for x, y in zip(o, a))}")
CONFIG["mismatch"] = [{"rec": r, "obs": list(o), "ann": list(a) if a else None} for r, o, a in mism]
CONFIG["excluded_in_q7b"] = EXCL17
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【E-A】 비트·RR 적재 + 코호트 구성
#   ⚠️ beat 배열이 크다(수백 MB). 한 번만 올리고 레코드별로 슬라이스한다.
d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"svdb_data5 와 예측 캐시 길이 불일치 {int(keep.sum())} vs {len(Y)}"
BEAT = np.asarray(d5["beat"])[keep]
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【E-A】 코호트")
run.log("=" * 100)
run.log(f"  비트 {BEAT.shape} · RR {PRE.shape} · 레코드 {len(ALLR)}개")
_pv = np.array([float((Y[REC == r] == IDX_S).mean()) for r in ALLR])
run.log(f"  S 유병률 — 중앙 {np.median(_pv):.4f} · 최대 {_pv.max():.4f}"
        f" · **> 0.5 인 개체 {int((_pv > 0.5).sum())}개**")
run.log(f"  라벨은 **평가에만** 쓴다. 앵커 구성은 라벨을 보지 않는다(다음 셀에서 검증).")
CONFIG["cohort"] = dict(n_records=len(ALLR), n_beats=int(len(Y)),
                        prev_median=float(np.median(_pv)), prev_max=float(_pv.max()),
                        n_prev_gt_half=int((_pv > 0.5).sum()))
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【E-B】 ★ 전수 재채점 — 기준 4종 × 전 개체
#
#   기준 넷 (전부 **같은 거리형 특징** ‖b − ref‖ 를 P영역에서 잰다):
#     · maj    다수결 = 레코드 중앙 (현행 `_medref`)      ← 다수가 S 면 끌려간다
#     · big    무감독 큰 군 중앙                            ← 다수결의 클러스터판
#     · anchor 무감독 **긴 RR 군** 중앙                     ← ★ 처방 후보. 개수를 안 본다
#     · oracle N군 중앙                                     ← **라벨 사용** = 상한
#   대조: RR 위치형(`중앙 − pre`) — 기준 불변이라 반전 불가 (Q7-D 에서 확인)
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans

def seg_dist(B, ref, seg):
    s0, s1 = seg
    d = B[:, :, s0:s1] - ref[None, :, s0:s1]
    return np.sqrt((d ** 2).sum(axis=(1, 2)))

def boot_auroc(tt, sc, seed, nb):
    """레코드 **내부**(비트) 부트스트랩 → 그 개체 AUROC 의 (lo, hi, SE).
    ★ 이게 있어야 '이 개체는 정말 뒤집혔나' 를 표본수와 함께 판정할 수 있다.
    ★ SE 를 함께 돌려주는 이유: 관문은 **본페로니 보정**된 문턱을 쓴다. 개체 수만큼
      검정을 반복하므로 개별 95% CI 로는 n×2.5% 만큼 오발한다(픽스처 실측)."""
    rng = np.random.RandomState(seed)
    v = []
    for _ in range(nb):
        j = rng.randint(0, len(tt), len(tt)); tj = tt[j]
        if 0 < tj.sum() < len(tj):
            v.append(roc_auc_score(tj.astype(int), sc[j]))
    if len(v) < 20:
        return float("nan"), float("nan"), float("nan")
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)),
            float(np.std(v, ddof=1)))

def anchor_split(Bp, pre_v, seed, min_clus):
    """P 영역 k-means 2군 → (lab, 큰 군, 긴RR 군, 중앙RR0, 중앙RR1).

    ★ **라벨을 인자로 받지 않는다.** 이 함수 안에서 정답(y/t)을 볼 방법이 없다 —
      그게 '무감독 앵커' 주장의 전부다. 픽스처가 이 함수 본문을 정적으로 검사한다.
    ★ 쓰는 사전지식은 **「이소성 박동은 이르다」 하나**. 이건 형태 지식이 아니라
      **리듬 지식**이다 → 늦은 이소성(이탈박동)에서는 틀린다.
    """
    lab = KMeans(2, n_init=10, random_state=seed).fit_predict(Bp.reshape(len(Bp), -1))
    n0, n1 = int((lab == 0).sum()), int((lab == 1).sum())
    if min(n0, n1) < min_clus:
        return None, f"군 크기 {min(n0, n1)} < {min_clus}"
    m0, m1 = float(np.median(pre_v[lab == 0])), float(np.median(pre_v[lab == 1]))
    if m0 == m1:
        return None, "두 군의 중앙 RR 동률 — 긴RR 앵커를 정의할 수 없다"
    big = 0 if n0 >= n1 else 1
    longrr = 0 if m0 > m1 else 1
    return (lab, big, longrr, m0, m1), None

run.log("\n" + "=" * 100)
run.log("【E-B】 전수 재채점 — 기준 4종")
run.log("=" * 100)
PER, SKIP = {}, []
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    if len(mm) < MIN_BEATS:
        SKIP.append((int(r), f"비트 {len(mm)} < {MIN_BEATS}")); continue
    if int(tt.sum()) < GMIN or bool(tt.all()):
        SKIP.append((int(r), f"S {int(tt.sum())} · N {int((~tt).sum())} — 채점 불가")); continue
    Bm, pre_m = BEAT[mm], PRE[mm]
    sp, why = anchor_split(Bm[:, :, P_SEG[0]:P_SEG[1]], pre_m, SEED0, MIN_CLUS)
    if sp is None:
        SKIP.append((int(r), f"앵커 퇴화 — {why}")); continue
    lab, big, longrr, m0, m1 = sp
    row = dict(n=int(len(mm)), pos=int(tt.sum()), prev=float(tt.mean()),
               s_frac_big=float(tt[lab == big].mean()),
               s_frac_anchor=float(tt[lab == longrr].mean()),
               big_is_anchor=bool(big == longrr),
               rr_long=float(max(m0, m1)), rr_short=float(min(m0, m1)),
               rr_sep=float(abs(m0 - m1) / max(np.median(pre_m), 1e-9)))
    REFS = {"maj": np.median(Bm, axis=0),
            "big": np.median(Bm[lab == big], axis=0),
            "anchor": np.median(Bm[lab == longrr], axis=0)}
    if int((~tt).sum()) >= MIN_CLUS:
        REFS["oracle"] = np.median(Bm[~tt], axis=0)
    for k_, ref in REFS.items():
        sc_ = seg_dist(Bm, ref, P_SEG)
        a_ = float(roc_auc_score(tt.astype(int), sc_))
        row[k_] = a_; row[k_ + "_info"] = float(max(a_, 1 - a_))
        if k_ in ("maj", "anchor"):        # 관문이 쓰는 두 기준만 CI 를 낸다(비용)
            (row[k_ + "_lo"], row[k_ + "_hi"],
             row[k_ + "_se"]) = boot_auroc(tt, sc_, SEED0 + int(r), NB_REC)
    for k_ in ("maj", "big", "anchor", "oracle"):
        row.setdefault(k_, float("nan")); row.setdefault(k_ + "_info", float("nan"))
    for k_ in ("maj", "anchor"):
        for sfx in ("_lo", "_hi", "_se"):
            row.setdefault(k_ + sfx, float("nan"))
    # 부지표: 전파형 축 (Q7-D 에서 다수결 0.0856 으로 더 심하게 뒤집혔던 축)
    row["full_maj"] = float(roc_auc_score(tt.astype(int), seg_dist(Bm, REFS["maj"], FULL_SEG)))
    row["full_anchor"] = float(roc_auc_score(tt.astype(int), seg_dist(Bm, REFS["anchor"], FULL_SEG)))
    # 대조: RR 위치형 (기준 불변)
    row["rr"] = float(roc_auc_score(tt.astype(int), np.median(pre_m) - pre_m))
    PER[int(r)] = row

run.log(f"  채점 {len(PER)}개체 · **제외 {len(SKIP)}개체** (조용히 빼지 않는다 — 전부 아래에)")
for r, why in SKIP:
    run.log(f"    #{r}  {why}")
if not PER:
    raise AssetError("채점된 개체가 없다 — 자산·문턱을 확인할 것")

RS = sorted(PER)
A = {k: np.array([PER[r][k] for r in RS]) for k in
     ("maj", "big", "anchor", "oracle", "rr", "full_maj", "full_anchor", "prev", "pos")}
run.log(f"\n  {'기준':<26}{'매크로':>9}{'SD':>9}{'중앙':>9}{'최소':>9}{'<0.5 개체':>11}")
NM = {"rr": "RR·위치형(기준 불변)", "maj": "형태·다수결(_medref)", "big": "형태·무감독 큰 군",
      "anchor": "형태·긴RR 앵커 ★", "oracle": "형태·오라클(라벨=상한)"}
for k in ("rr", "maj", "big", "anchor", "oracle"):
    v = A[k]; f = np.isfinite(v)
    run.log(f"  {NM[k]:<26}{np.nanmean(v):>9.4f}{np.nanstd(v[f], ddof=1):>9.4f}"
            f"{np.nanmedian(v):>9.4f}{np.nanmin(v):>9.4f}{int((v[f] < 0.5).sum()):>11}")
run.log(f"  {'(부지표) 전파형·다수결':<26}{np.nanmean(A['full_maj']):>9.4f}")
run.log(f"  {'(부지표) 전파형·앵커':<26}{np.nanmean(A['full_anchor']):>9.4f}")
run.log("\n  ⚠️ 이 표의 값은 **무감독 거리 특징**이다. Q7-B 학습 모델의 0.8842 와 비교하지 않는다.")
CONFIG["per_record"] = {str(r): PER[r] for r in RS}
CONFIG["skipped"] = [{"rec": r, "why": w} for r, w in SKIP]
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【E-C】 관문 판정 — **전부 짝지은 비교**
#   개체 간 이질성(Q7-B′ SD 0.157)이 크므로 비짝지은 비교는 검정력을 버린다.
#   같은 환자에서 기준만 바꿔 뺀다 → 이질성이 상쇄된다(Q7-D 【D-E】 와 같은 논리).
def boot_diff(a, b, seed, nb=NB_BOOT):
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)])
    return float(d.mean()), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)), len(d)

run.log("\n" + "=" * 100)
run.log("【E-C】 관문")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

# ── E1 개선
m1, lo1, hi1, n1_ = boot_diff(A["anchor"], A["maj"], SEED0 + 1)
DIFF["E1"] = dict(mean=m1, lo=lo1, hi=hi1, n=n1_)
g_("E1", decide(lo1, hi1, 0.0, ">"),
   f"짝지은 (앵커 − 다수결) **{m1:+.4f}** [{lo1:+.4f}, {hi1:+.4f}] · n={n1_}"
   f" · 좋아진 개체 {int((A['anchor'] > A['maj']).sum())}/{n1_}")

# ── E2 반전 개체 0
#   ★ '뒤집혔다' 는 **개체별 불확실성 + 다중검정 보정**으로 판정한다.
#     ① 점추정에 고정 문턱(INFO_MIN)을 걸면 S 가 십수 개뿐인 개체가 우연히 0.34 로
#        찍혀도 '반전' 이 된다.
#     ② 개별 95% CI 로 바꿔도 부족하다 — 개체마다 검정을 반복하므로 n개면 n×2.5%
#        만큼 오발한다. 합성 null 코호트 14개체에서 실제로 1개(#901, CI 상한 0.491)가
#        걸렸다. **신호가 0인 코호트인데도.**
#     → **본페로니**: 개체 수 n 에 대해 양측 α/n 문턱을 쓴다.
Z_BONF = float(stats.norm.ppf(1 - 0.05 / (2 * max(len(RS), 1))))
run.log(f"  다중검정 보정 — 개체 {len(RS)}개 · 본페로니 z = {Z_BONF:.3f} (개별 95% 는 1.960)")
def sig_inv(r, k):
    a_, s_ = PER[r][k], PER[r][k + "_se"]
    if not np.isfinite(a_) or not np.isfinite(s_):
        return False
    if s_ <= 0:                       # 완전 분리 — SE 0 이면 점추정이 곧 결론
        return a_ < 0.5
    return a_ + Z_BONF * s_ < 0.5
inv_maj = [r for r in RS if sig_inv(r, "maj")]
inv_anc = [r for r in RS if sig_inv(r, "anchor")]
pt_maj = [r for r in RS if PER[r]["maj"] < 0.5 and PER[r]["maj_info"] >= INFO_MIN]
pt_anc = [r for r in RS if PER[r]["anchor"] < 0.5 and PER[r]["anchor_info"] >= INFO_MIN]
run.log(f"  (참고) 다수결 기준 **유의 반전** {len(inv_maj)}개: "
        + (", ".join(f"#{r}({PER[r]['maj']:.3f} ±{PER[r]['maj_se']:.3f})"
                     for r in inv_maj) or "없음"))
run.log(f"  (참고) 점추정만 반전 — 다수결 {len(pt_maj)}개 · 앵커 {len(pt_anc)}개"
        f"  (표본수를 안 보는 셈법이라 관문에 쓰지 않는다)")
g_("E2", "✅ 지지" if len(inv_anc) == 0 else "❌ 기각",
   f"앵커 기준 **유의 반전** 개체 **{len(inv_anc)}개**: "
   + (", ".join(f"#{r}({PER[r]['anchor']:.3f} ±{PER[r]['anchor_se']:.3f}"
                f" · 유병률 {PER[r]['prev']:.3f})" for r in inv_anc) or "없음"))

# ── E3 정상 개체 비열등 (유병률 ≤ 0.5)
nm = np.array([PER[r]["prev"] <= 0.5 for r in RS])
m3, lo3, hi3, n3_ = boot_diff(A["anchor"][nm], A["maj"][nm], SEED0 + 3)
DIFF["E3"] = dict(mean=m3, lo=lo3, hi=hi3, n=n3_)
g_("E3", decide(lo3, hi3, -NI_NORMAL, ">"),
   f"유병률 ≤ 0.5 인 {n3_}개체에서 (앵커 − 다수결) **{m3:+.4f}** [{lo3:+.4f}, {hi3:+.4f}]"
   f"  vs 비열등 여유 −{NI_NORMAL}")
worst = sorted((r for r in RS if PER[r]["prev"] <= 0.5),
               key=lambda r: PER[r]["anchor"] - PER[r]["maj"])[:5]
run.log("    가장 손해 본 정상 개체 5: "
        + ", ".join(f"#{r}({PER[r]['maj']:.3f}→{PER[r]['anchor']:.3f})" for r in worst))

# ── E4 오라클 회복률
om = np.isfinite(A["oracle"])
m4, lo4, hi4, n4_ = boot_diff(A["oracle"][om], A["anchor"][om], SEED0 + 4)
DIFF["E4"] = dict(mean=m4, lo=lo4, hi=hi4, n=n4_)
g_("E4", decide(lo4, hi4, ORACLE_MAX, "<"),
   f"(오라클 − 앵커) **{m4:+.4f}** [{lo4:+.4f}, {hi4:+.4f}] · n={n4_} vs 상한 {ORACLE_MAX}")

# ── E5 RR 위치형 대비 (★ '형태로 갈아타면 되냐' 에 대한 답)
m5, lo5, hi5, n5_ = boot_diff(A["anchor"], A["rr"], SEED0 + 5)
DIFF["E5"] = dict(mean=m5, lo=lo5, hi=hi5, n=n5_)
g_("E5", decide(lo5, hi5, -NI_RR, ">"),
   f"(앵커 형태 − RR 위치형) **{m5:+.4f}** [{lo5:+.4f}, {hi5:+.4f}] vs 비열등 여유 −{NI_RR}")

# ── 최대 기여 개체 (R11-c) — 한 개체가 개선을 다 만들었나
d1 = A["anchor"] - A["maj"]
tot = float(np.abs(d1).sum())
top = sorted(range(len(RS)), key=lambda i: -abs(d1[i]))[:5]
run.log(f"\n  개선 기여 상위 5 (|Δ| 합 {tot:.3f} 중):")
for i in top:
    run.log(f"    #{RS[i]}  Δ {d1[i]:+.4f}  ({abs(d1[i])/max(tot,1e-9)*100:.1f}%)"
            f"  유병률 {PER[RS[i]]['prev']:.3f} · 앵커군 S비율 {PER[RS[i]]['s_frac_anchor']:.3f}")
m1b, lo1b, hi1b, n1b = boot_diff(np.delete(A["anchor"], top[0]), np.delete(A["maj"], top[0]), SEED0 + 6)
#   ★ 애초에 개선이 없으면(|Δ| 합 ≈ 0) '한 개체 이야기' 라는 문구 자체가 오독이다.
_msg = ("← 애초에 기준 간 차이가 없다(|Δ| 합 ≈ 0)" if tot < 1e-6 else
        "← 개선이 살아남는다" if lo1b > 0 else "← **한 개체 이야기였다**")
run.log(f"  최대 기여 개체 #{RS[top[0]]} 제외 → **{m1b:+.4f}** [{lo1b:+.4f}, {hi1b:+.4f}]   {_msg}")
DIFF["E1_ex_top"] = dict(mean=m1b, lo=lo1b, hi=hi1b, n=n1b, dropped=int(RS[top[0]]))

# ── 산포·검정력 (R15 · R17)
sd_a = float(np.nanstd(A["anchor"], ddof=1)); sd_d = float(np.nanstd(d1[np.isfinite(d1)], ddof=1))
_ratio = ("측정 불가(둘 중 하나가 0)" if min(sd_a, sd_d) < 1e-9
          else f"짝지으면 {sd_a/sd_d:.1f}배 좁다 → 검정력을 번다" if sd_d < sd_a
          else f"짝지어도 안 좁아진다({sd_a/sd_d:.1f}배) — 기준별 성능이 개체마다 따로 논다")
run.log(f"\n  개체 간 SD — 앵커 {sd_a:.4f} · **짝지은 차 {sd_d:.4f}**  ({_ratio})")
run.log("    ⚠️ 채점된 부분집합의 산포는 모집단의 **하한**이다 (R17)")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
CONFIG["inverted"] = dict(maj=[int(r) for r in inv_maj], anchor=[int(r) for r in inv_anc],
                          point_maj=[int(r) for r in pt_maj], point_anchor=[int(r) for r in pt_anc])
CONFIG["sd"] = dict(anchor=sd_a, paired=sd_d, z_bonferroni=Z_BONF)
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【E-D】 앵커가 지는 곳 — **사후 탐색(관문 아님)**
#   ★ 여기서 나온 조건으로 위 관문을 다시 매기지 않는다. 다음 실험의 **사전등록 재료**다.
run.log("\n" + "=" * 100)
run.log("【E-D】 사후 탐색 — 앵커 실패의 구조 (관문 아님)")
run.log("=" * 100)

# ① 앵커가 '잘못된 군' 을 골랐나 — 라벨로 사후 확인만
bad = [r for r in RS if PER[r]["s_frac_anchor"] > 0.5]
run.log(f"  앵커가 **S 우세 군**을 기저로 고른 개체 {len(bad)}개: "
        + (", ".join(f"#{r}(S비율 {PER[r]['s_frac_anchor']:.2f} · AUROC {PER[r]['anchor']:.3f})"
                     for r in bad) or "없음"))
run.log("    → 이게 0 이 아니면 「이소성은 이르다」 가정이 그 개체에서 깨진 것이다"
        " (이탈박동 등 **늦은 이소성**).")

# ② 두 군의 RR 분리도와 성능의 관계
sep = np.array([PER[r]["rr_sep"] for r in RS])
rho_s, p_s = stats.spearmanr(sep, A["anchor"])
run.log(f"\n  두 군 중앙 RR 분리도 vs 앵커 AUROC — rho={rho_s:+.3f} p={p_s:.4f}")
for lo_, hi_ in ((0.0, 0.05), (0.05, 0.15), (0.15, 0.30), (0.30, 9.9)):
    s_ = (sep >= lo_) & (sep < hi_)
    if s_.sum():
        run.log(f"    분리도 [{lo_:.2f},{hi_:.2f}) — {int(s_.sum()):>2}개체 · "
                f"앵커 {A['anchor'][s_].mean():.4f} · 다수결 {A['maj'][s_].mean():.4f} · "
                f"Δ {(A['anchor'][s_]-A['maj'][s_]).mean():+.4f}")

# ③ 유병률 층화 (R11-c — 총계를 바꾸지 말고 층으로 보고한다)
run.log("")
for lo_, hi_ in ((0.0, 0.05), (0.05, 0.20), (0.20, 0.50), (0.50, 1.01)):
    s_ = (A["prev"] >= lo_) & (A["prev"] < hi_)
    if s_.sum():
        run.log(f"  유병률 [{lo_:.2f},{hi_:.2f}) — {int(s_.sum()):>2}개체 · "
                f"RR {A['rr'][s_].mean():.4f} · 다수결 {A['maj'][s_].mean():.4f} · "
                f"앵커 {A['anchor'][s_].mean():.4f} · 오라클 {np.nanmean(A['oracle'][s_]):.4f}")
CONFIG["posthoc"] = dict(
    anchor_picked_s_cluster=[int(r) for r in bad],
    rho_sep=[float(rho_s), float(p_s)],
    note="사후 탐색 — 관문 재판정 금지. 다음 실험 사전등록 재료")
run.save_json("config", CONFIG)

# ── 결론 문장을 **관문 조합으로만** 만든다
ok = lambda k: VERD.get(k, "").startswith("✅")
run.log("\n" + "=" * 100)
if ok("E1") and ok("E2") and ok("E3"):
    run.log("  ★ 처방 성립 — 기준을 다수에서 떼면 반전이 사라지고 정상 개체도 안 상한다 (R19)")
elif ok("E1") and ok("E2") and not ok("E3"):
    run.log("  ⚠️ 반전은 잡았지만 **정상 개체를 깎았다** — 조건부 적용(분리도 문턱)이 필요하다")
elif not ok("E2"):
    run.log("  ⛔ 앵커로도 반전이 남는다 — 레코드 내부만으로는 기준을 못 잡는 개체가 있다."
            " **교차환자 N 분포** 같은 절대 기준이 필요하다")
else:
    run.log("  ⚠️ 개선이 확정되지 않았다 — 아래 관문표를 그대로 읽을 것")
if not ok("E5"):
    run.log(f"  · E5 {VERD.get('E5','(미실행)')} = **형태로 갈아타는 게 답이 아니다.**"
            " 앵커의 값어치는 형태 축을 **되살린 것**이지 RR 을 이긴 게 아니다."
            " 기준 문제는 축을 바꿔서 못 푼다")
run.log("=" * 100)

In [ ]:
# CELL 7 — 그림 + 마무리
#  ★ Colab 기본 폰트에 한글이 없어 □ 로 깨진다. 그림 라벨은 ASCII 로 쓴다.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
rs_ = np.array(RS)

ax[0].scatter(A["maj"], A["anchor"], s=34, alpha=.8,
              c=np.where(A["prev"] > 0.5, "crimson", "C0"))
ax[0].plot([0, 1], [0, 1], color="gray", ls="--", lw=1)
ax[0].axhline(.5, color="crimson", ls=":", lw=1); ax[0].axvline(.5, color="crimson", ls=":", lw=1)
for i in np.where((A["maj"] < 0.5) | (A["anchor"] < 0.5))[0]:
    ax[0].annotate(f"#{rs_[i]}", (A["maj"][i], A["anchor"][i]), fontsize=8,
                   xytext=(5, 4), textcoords="offset points")
ax[0].set_xlabel("AUROC, majority reference (_medref)")
ax[0].set_ylabel("AUROC, LONG-RR anchor")
ax[0].set_title(f"(1) per-record: majority vs anchor  (n={len(rs_)}, red = prevalence>0.5)")
ax[0].grid(alpha=.3); ax[0].set_xlim(-.02, 1.02); ax[0].set_ylim(-.02, 1.02)

d_ = A["anchor"] - A["maj"]
ax[1].scatter(A["prev"], d_, s=34, alpha=.8)
ax[1].axhline(0, color="gray", ls="--", lw=1)
ax[1].axhline(-NI_NORMAL, color="crimson", ls=":", lw=1, label=f"non-inferiority -{NI_NORMAL}")
ax[1].axvline(.5, color="gray", ls=":", lw=1)
ax[1].set_xlabel("S prevalence"); ax[1].set_ylabel("delta AUROC (anchor - majority)")
ax[1].set_title(f"(2) who gains, who loses   paired mean {DIFF['E1']['mean']:+.4f}")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)

KS = ["rr", "maj", "big", "anchor", "oracle"]
LB = ["RR positional\n(reference-free)", "morph\nmajority(_medref)", "morph\nunsup BIG",
      "morph\nLONG-RR anchor", "morph\nORACLE (labels)"]
CL = ["C7", "C3", "C3", "C0", "C2"]
ax[2].bar(range(len(KS)), [float(np.nanmean(A[k])) for k in KS], color=CL)
ax[2].axhline(0.5, color="crimson", ls="--", lw=1)
ax[2].set_xticks(range(len(KS))); ax[2].set_xticklabels(LB, fontsize=7)
ax[2].set_ylabel("macro AUROC (unsupervised distance)")
ax[2].set_title("(3) macro by reference   green = ORACLE upper bound")
ax[2].grid(alpha=.3, axis="y"); ax[2].set_ylim(0, 1.05)
plt.tight_layout(); run.save_fig("q7e_anchor_sweep", fig); plt.show()

run.log("\n" + "=" * 100)
run.log("관문 요약")
run.log("=" * 100)
for k in ("E1", "E2", "E3", "E4", "E5"):
    run.log(f"  {k:<4}{VERD.get(k, '(미실행)')}")
run.finish({"verdicts": VERD, "diffs": DIFF,
            "macro": {k: float(np.nanmean(A[k])) for k in KS},
            "n_scored": len(RS), "n_skipped": len(SKIP)})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-anchor-sweep`")